## Microsoft Entra ID 개요

Microsoft Entra ID(이전 Azure Active Directory)는 Microsoft의 클라우드 기반 ID 및 액세스 관리 서비스입니다. Microsoft 365, Azure 및 
수천 개의 기타 SaaS 애플리케이션을 위한 중앙 IdP 역할을 합니다.

주요 기능:
* **Single Sign-On(SSO)**: 한 번 인증하여 여러 애플리케이션에 액세스
* **Multi-Factor Authentication(MFA)**: 추가 검증 방법을 통한 보안 강화
* **Conditional Access**: 사용자, 디바이스, 위치, 위험을 기반으로 하는 정책 기반 액세스 제어
* **애플리케이션 통합**: OAuth 2.0, OpenID Connect, SAML 같은 최신 인증 프로토콜 지원

## Amazon Bedrock AgentCore Gateway 개요

Bedrock AgentCore Gateway를 사용하면 인프라나 호스팅을 관리하지 않고도 기존 API와 Lambda 함수를 완전관리형 MCP server로 전환할 수 있습니다. 기존 API의 OpenAPI spec 또는 Smithy 모델을 가져오거나 도구 앞단에 Lambda 함수를 추가할 수 있습니다. Gateway는 이러한 모든 도구에 일관된 Model Context Protocol(MCP) 인터페이스를 제공합니다. Gateway는 inbound 요청과 대상 리소스로 나가는 연결 모두에 안전한 액세스 제어를 적용하기 위해 이중 인증 모델을 사용합니다. 이 프레임워크는 Gateway 대상에 액세스하려는 사용자를 검증하고 권한을 부여하는 Inbound Auth와, 인증된 사용자를 대신해 Gateway가 backend 리소스에 안전하게 연결하도록 하는 Outbound Auth로 구성됩니다. 두 인증 메커니즘은 사용자와 대상 리소스 사이에 안전한 연결을 만들며 IAM 자격 증명과 OAuth 기반 인증 흐름을 모두 지원합니다. Gateway는 MCP의 Streamable HTTP transport 연결을 지원합니다.

Amazon Bedrock AgentCore Gateway에 대한 자세한 내용은 다음을 참조하세요.
- https://github.com/awslabs/amazon-bedrock-agentcore-samples/tree/main/06-workshops/02-AgentCore-gateway
- https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway.html

## 학습 목표
Microsoft Entra ID를 AgentCore Identity의 IdP로 사용하여 소비 애플리케이션이 보호된 Amazon Bedrock AgentCore Gateway 리소스에 액세스하도록 권한을 부여할 수 있습니다. 이 Notebook에서는 Amazon Bedrock AgentCore Gateway의 Inbound Auth에 Entra ID를 사용하는 방법을 살펴봅니다.

## 학습 목표 1: AgentCore Gateway에서 사용할 Entra ID 설정

### 1단계: Entra ID Tenant 설정
Entra ID tenant는 조직을 나타내는 전용 Microsoft Entra ID 인스턴스입니다. Microsoft 클라우드에 격리된 조직 디렉터리라고 생각할 수 있습니다.

주요 특성:
* **고유한 ID**: 각 tenant에는 고유한 domain이 있습니다(예: yourcompany.onmicrosoft.com).
* **격리된 경계**: 한 tenant의 사용자, 그룹, 애플리케이션은 다른 tenant와 분리됩니다.
* **관리 제어**: tenant 관리자가 사용자, 보안 정책, app registration을 관리합니다.
* **다중 Domain 지원**: 기본 .onmicrosoft.com domain과 함께 custom domain을 포함할 수 있습니다.

실제 적용:
OAuth 통합을 위해 Entra ID에 애플리케이션을 등록하면 특정 tenant 내부에 등록됩니다. 이후 해당 tenant의 사용자는 조직 자격 증명으로 애플리케이션에 인증할 수 있습니다.

AgentCore 통합에는 다음 항목이 필요합니다.
* **Tenant ID**: Entra ID 인스턴스의 고유 식별자
* **Application Registration**: tenant에 등록된 앱
* **적절한 권한**: 애플리케이션에 구성된 액세스 권한

이 tenant 기반 모델은 인증과 권한 부여가 조직의 보안 경계 안에서 유지되도록 합니다.

tenant 생성 단계는 https://learn.microsoft.com/en-us/entra/fundamentals/create-new-tenant 에서 확인할 수 있습니다.

참고:
1. Microsoft Entra ID는 AWS 서비스가 아닙니다. 비용 관련 정보는 Microsoft Entra ID 문서를 참조하세요.
2. 다음 단계의 화면은 변경될 수 있습니다. Entra ID 애플리케이션 설정에 대한 최신 지침은 Microsoft Entra ID 문서를 참조하세요.

In [ ]:
import os

os.environ["tenant_id"] = "REPLACE_ME"  # Entra ID의 Tenant ID로 교체

### 2단계: 사용할 API 정의
1. portal.azure.com으로 이동하여 화면 상단의 검색창에서 "Entra ID"를 검색합니다.
<img src="images/entraid.jpg" width="75%">
2. manage --> App Registrations로 이동합니다.
<img src="images/app.registration.png" width="75%">
3. "New Registration"을 클릭하고 세부 정보를 입력합니다. multi-tenant 옵션을 선택합니다.
- redirect URL은 설정하지 않습니다.   
<img src="images/setup.api.png" width="75%">
4. Manage --> Expose an API에서 API를 노출합니다.
<img src="images/api.expose.png" width="75%"/>
5. API의 app role을 생성합니다. M2M 설정이므로 scope는 추가하지 않습니다.
<img src="images/weather.app.role.png" width="75%"/>

In [ ]:
# app_id_url은 "App registration" --> "All Applications" --> 방금 생성한 client 선택 --> "Expose a API"의 "Application ID URI"
os.environ["app_id_uri"] = (
    "api://3dXXXXXX-CCCC-VVVV-BBBB-NNNNNN885f25"  # "weather_service"에 설정한 API URL
)

### 3단계: Entra Client 애플리케이션 생성
1. portal.azure.com으로 이동하여 화면 상단의 검색창에서 "Entra ID"를 검색합니다.

<img src="images/entraid.jpg" width="75%">

2. manage --> App Registrations로 이동합니다.

<img src="images/app.registration.png" width="75%">

3. "New Registration"을 클릭하고 세부 정보를 입력합니다. multi-tenant 옵션을 선택합니다.
- redirect URL은 설정하지 않습니다.

<img src="images/client.register.png" width="75%"/>

4. client secret을 생성합니다. AgentCore에서 사용할 client secret과 client ID를 복사합니다.

<img src="images/client.secret.png" width="75%">

5. API permissions로 이동하여 앞에서 생성한 API의 권한을 요청합니다. "API Permissions" --> "Add a Permission" --> "APIs my organziation uses"에서 1단계에 생성한 API를 검색합니다.

<img src="images/api.permissions.png" width="75%">

6. API 사용을 위한 admin consent를 부여합니다.

7. Entra ID 정보를 사용해 환경 변수를 설정합니다.

In [ ]:
import os

# Tenant ID: "App registration" --> "All Applications" --> 방금 생성한 client 선택 --> "Overview" --> "Directory (tenant) ID"
os.environ["tenant_id"] = "bc24XXXX-CCCC-VVVV-BBBB-NNNNb5df1f19"

# Client ID: "App registration" --> "All Applications" --> 방금 생성한 client 선택 --> "Overview" --> "Application (client) ID"
os.environ["client_id"] = (
    "08XXXXXX-CCCC-VVVV-BBBB-NNNNNNd86cd2"  # "weather_service_client"의 Client ID로 교체
)

# 앞 단계에서 저장한 secret
os.environ["client_secret"] = (
    "muCCCCCVVVVVBBBBBNNNNN3dY6qdlL"  # "weather_service_client"의 Client secret으로 교체
)

## 학습 목표 2: AgentCore Gateway 및 Lambda 대상 설정

### 1단계: Entra ID와 함께 사용할 Lambda 함수 생성
1. Lambda 함수 코드로 사용할 Python 파일을 생성합니다. 호출되는 도구 이름을 `context` 객체에서 가져와 Lambda 함수에서 사용하는 방식을 확인하세요.

In [ ]:
import boto3
import zipfile
from boto3.session import Session
import time

boto_session = Session()
sts = boto3.client("sts")
region = boto_session.region_name
account_id = sts.get_caller_identity().get("Account")

In [ ]:
%%writefile lambda_function.py
def lambda_handler(event, context):
    print(f"Event: {event}")
    print(f"Context: {context}")
    extended_tool_name = context.client_context.custom["bedrockAgentCoreToolName"]
    resource = extended_tool_name.split("___")[1]

    print(resource)
    city = event.get("city")
    print(city)
    if resource == "weather_check":
        return f"Weather in {city} is bright and sunny!"
    elif resource == "directions":
        return f"Take I5 south all the way to {city} downtown"

2. Lambda 함수를 생성합니다.

In [ ]:
lambda_client = boto3.client("lambda", region_name=region)
with zipfile.ZipFile("lambda_function.zip", "w") as zip_file:
    zip_file.write("lambda_function.py", "lambda_function.py")

with open("lambda_function.zip", "rb") as zip_file:
    zip_content = zip_file.read()

In [ ]:
iam_client = boto3.client("iam", region_name=region)

trust_policy = """{
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {
                "Service": "lambda.amazonaws.com"
            },
            "Action": "sts:AssumeRole"
        }
    ]
}
"""

policy = """{
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "logs:CreateLogGroup",
                "logs:CreateLogStream",
                "logs:PutLogEvents"
            ],
            "Resource": "arn:aws:logs:*:*:*"
        }
    ]
}
"""

response = iam_client.create_role(RoleName="lambda-role", AssumeRolePolicyDocument=trust_policy)

iam_client.put_role_policy(PolicyDocument=policy, PolicyName="lambda-policy", RoleName="lambda-role")

lambda_role_arn = response["Role"]["Arn"]

# 역할이 전파될 때까지 대기
time.sleep(10)

response = lambda_client.create_function(
    FunctionName="m2m-entra-lambda",
    Runtime="python3.12",
    Role=lambda_role_arn,
    Handler="lambda_function.lambda_handler",
    Code={"ZipFile": zip_content},
)

In [ ]:
lambda_arn = response["FunctionArn"]

In [ ]:
lambda_arn

### 2단계: Inbound 보안이 적용된 Amazon Bedrock AgentCore Gateway 생성

In [ ]:
iam_client = boto3.client("iam")

trust_policy = """{
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {
                "Service": "bedrock-agentcore.amazonaws.com"
            },
            "Action": "sts:AssumeRole"
        }
    ]
}
"""

# trust policy가 포함된 역할 생성
response = iam_client.create_role(RoleName="bedrock-agent-lambda-role", AssumeRolePolicyDocument=trust_policy)

permission = (
    """{
    "Version": "2012-10-17",
    "Statement": [
        {
            "Action": [
                "lambda:InvokeFunction"
            ],
            "Resource": [
                "%s"
            ],
            "Effect": "Allow",
            "Sid": "InvokeFunction"
        }
    ]
}
"""
    % lambda_arn
)


# Lambda invoke policy 추가
iam_client.put_role_policy(
    RoleName="bedrock-agent-lambda-role",
    PolicyName="lambda-invoke-policy",
    PolicyDocument=permission,
)

role_arn = response["Role"]["Arn"]
print(f"Role ARN: {role_arn}")

In [ ]:
gateway_client = boto3.client(
    "bedrock-agentcore-control",
    region_name=region,
)

gateway_name = "m2m-entra-gateway"
auth_config = {
    "customJWTAuthorizer": {
        "allowedAudience": [os.environ["app_id_uri"]],
        "discoveryUrl": f"https://login.microsoftonline.com/{os.environ['tenant_id']}/.well-known/openid-configuration",
    }
}

In [ ]:
create_response = gateway_client.create_gateway(
    name=gateway_name,
    roleArn=role_arn,
    protocolType="MCP",
    authorizerType="CUSTOM_JWT",
    authorizerConfiguration=auth_config,
    description="Customer Support AgentCore Gateway",
)

In [ ]:
gateway_url = create_response["gatewayUrl"]
gateway_id = create_response["gatewayId"]

### 3단계: 방금 생성한 AgentCore Gateway에 Lambda 대상 추가

1. Lambda 함수를 통해 생성하는 실제 도구의 API specification입니다.

In [ ]:
api_spec = [
    {
        "name": "weather_check",
        "description": "Check the weather for a given City",
        "inputSchema": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "The city you want to get weather for",
                }
            },
            "required": ["city"],
        },
    },
    {
        "name": "directions",
        "description": "Search the web for directions to a city",
        "inputSchema": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "The city you want to get directions to",
                }
            },
            "required": ["city"],
        },
    },
]

In [ ]:
lambda_target_config = {
    "mcp": {
        "lambda": {
            "lambdaArn": lambda_arn,
            "toolSchema": {"inlinePayload": api_spec},
        }
    }
}

# Gateway 대상 생성
credential_config = [{"credentialProviderType": "GATEWAY_IAM_ROLE"}]

create_target_response = gateway_client.create_gateway_target(
    gatewayIdentifier=gateway_id,
    name="LambdaUsingSDK",
    description="Lambda Target using SDK",
    targetConfiguration=lambda_target_config,
    credentialProviderConfigurations=credential_config,
)

## 학습 목표 3: AgentCore Gateway에서 제공하는 도구를 에이전트에서 사용

### 1단계: 토큰을 가져와 payload 및 header 검토
1. access token을 가져와 AgentCore Gateway 액세스에 사용합니다.

In [ ]:
import requests
import json

TOKEN_URL = f"https://login.microsoftonline.com/{os.environ['tenant_id']}/oauth2/v2.0/token"
SCOPE = f"{os.environ['app_id_uri']}/.default"


def fetch_access_token(client_id, client_secret, token_url, scope):
    data = {
        "grant_type": "client_credentials",
        "client_id": client_id,
        "client_secret": client_secret,
        "scope": scope,
    }

    response = requests.post(
        token_url,
        data=data,
        headers={"Content-Type": "application/x-www-form-urlencoded"},
    )
    # print(response.text)
    return response.json()["access_token"]


access_token = fetch_access_token(os.environ["client_id"], os.environ["client_secret"], TOKEN_URL, SCOPE)

2. 토큰을 decode하여 내용을 확인합니다. "aud", "appid", "roles"가 앞에서 설정한 값과 일치하는지 확인하세요.

In [ ]:
import base64


def decode_jwt_token(token):
    # JWT를 부분별로 분리
    parts = token.split(".")

    # header 디코딩
    header = json.loads(base64.b64decode(parts[0] + "==").decode("utf-8"))

    # payload 디코딩
    payload = json.loads(base64.b64decode(parts[1] + "==").decode("utf-8"))

    return header, payload


# 사용 예
header, payload = decode_jwt_token(access_token)

print("Header:", json.dumps(header, indent=2))
print("Payload:", json.dumps(payload, indent=2))

# 특정 claim 확인
print(f"Audience: {payload.get('aud')}")
print(f"Issuer: {payload.get('iss')}")
print(f"Expires: {payload.get('exp')}")
print(f"Scopes: {payload.get('scp')}")
print(f"Roles: {payload.get('roles')}")

### 2단계: access token으로 AgentCore Gateway에서 사용 가능한 도구 목록 가져오기
아래와 유사한 도구 specification이 표시됩니다.   

<img src="images/tools.spec.png" width="50%"/>

In [ ]:
def list_tools(gateway_url, access_token):
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {access_token}",
    }

    payload = {"jsonrpc": "2.0", "id": "list-tools-request", "method": "tools/list"}

    response = requests.post(gateway_url, headers=headers, json=payload)
    return response.json()


tools = list_tools(gateway_url, access_token)
print(json.dumps(tools, indent=2))

### 3단계: MCP client를 생성하고 도구 목록을 가져와 Strands Agent에서 사용

In [ ]:
from mcp.client.streamable_http import streamablehttp_client
from strands.tools.mcp import MCPClient

# MCP client 설정
mcp_client = MCPClient(
    lambda: streamablehttp_client(
        gateway_url,
        headers={"Authorization": f"Bearer {access_token}"},
    )
)

In [ ]:
mcp_client.start()

In [ ]:
mcp_client.list_tools_sync()

In [ ]:
from strands import Agent

agent = Agent(tools=mcp_client.list_tools_sync())

#### 참고: 앞에서 정의한 Lambda 함수의 응답은 정적입니다. 따라서 prompt에 어떤 도시를 입력해도 에이전트의 응답은 매우 유사합니다.

In [ ]:
agent("What is the weather in San Diego?")

In [ ]:
agent("Give me directions to San Diego?")

## 마무리 및 정리
이 Notebook에서는 다음 내용을 알아보았습니다.
- OAuth Client Credential(M2M) flow를 제공하도록 Entra ID API와 애플리케이션 설정
- AgentCore Gateway 생성
- Lambda 함수를 생성하고 AgentCore Gateway의 대상으로 추가. Lambda 함수는 AgentCore Gateway를 통해 MCP 도구로 제공됨
- MCP client로 Gateway의 도구에 액세스하고 Strands Agent에 바인딩하여 사용자 질의 처리

#### 생성된 리소스

In [ ]:
lambda_arn, role_arn, gateway_id, lambda_role_arn, create_response["gatewayArn"]

In [ ]:
create_target_response["targetId"]

#### Gateway의 Lambda 대상 삭제

In [ ]:
gateway_client.delete_gateway_target(gatewayIdentifier=gateway_id, targetId=create_target_response["targetId"])

#### Gateway 삭제

In [ ]:
gateway_client.delete_gateway(gatewayIdentifier=gateway_id)

#### 생성한 Lambda 함수 삭제

In [ ]:
function_name = lambda_arn.split(":")[-1]
lambda_client.delete_function(FunctionName=function_name)

#### 생성한 역할 삭제

In [ ]:
role_name = lambda_role_arn.split("/")[-1]
inline = iam_client.list_role_policies(RoleName=role_name)
for policy_name in inline["PolicyNames"]:
    iam_client.delete_role_policy(RoleName=role_name, PolicyName=policy_name)
iam_client.delete_role(RoleName=role_name)

In [ ]:
role_name = role_arn.split("/")[-1]
inline = iam_client.list_role_policies(RoleName=role_name)
for policy_name in inline["PolicyNames"]:
    iam_client.delete_role_policy(RoleName=role_name, PolicyName=policy_name)
iam_client.delete_role(RoleName=role_name)